In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("/kaggle/input/medical-insurance-cost-prediction/medical_insurance.csv")

In [3]:
data.head()

,person_id,age,sex,region,urban_rural,income,education,marital_status,employment_status,household_size,...,liver_disease,arthritis,mental_health,proc_imaging_count,proc_surgery_count,proc_physio_count,proc_consult_count,proc_lab_count,is_high_risk,had_major_procedure
0,75722,52,Female,North,Suburban,22700.0,Doctorate,Married,Retired,3,...,0,1,0,1,0,2,0,1,0,0
1,80185,79,Female,North,Urban,12800.0,No HS,Married,Employed,3,...,0,1,1,0,0,1,0,1,1,0
2,19865,68,Male,North,Rural,40700.0,HS,Married,Retired,5,...,0,0,1,1,0,2,1,0,1,0
3,76700,15,Male,North,Suburban,15600.0,Some College,Married,Self-employed,5,...,0,0,0,1,0,0,1,0,0,0
4,92992,53,Male,Central,Suburban,89600.0,Doctorate,Married,Self-employed,2,...,0,1,0,2,0,1,1,0,1,0


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 54 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   person_id                    100000 non-null  int64  
 1   age                          100000 non-null  int64  
 2   sex                          100000 non-null  object 
 3   region                       100000 non-null  object 
 4   urban_rural                  100000 non-null  object 
 5   income                       100000 non-null  float64
 6   education                    100000 non-null  object 
 7   marital_status               100000 non-null  object 
 8   employment_status            100000 non-null  object 
 9   household_size               100000 non-null  int64  
 10  dependents                   100000 non-null  int64  
 11  bmi                          100000 non-null  float64
 12  smoker                       100000 non-null  object 
 13  

In [5]:
data["alcohol_freq"].describe()

count          69917
unique             3
top       Occasional
freq           45078
Name: alcohol_freq, dtype: object

In [6]:
data["alcohol_freq"].value_counts()

alcohol_freq
Occasional    45078
Weekly        19833
Daily          5006
Name: count, dtype: int64

In [7]:
data["alcohol_freq"].fillna(value = "Occasional", inplace = True, axis = 0)

/tmp/ipykernel_13/1427768181.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data["alcohol_freq"].fillna(value = "Occasional", inplace = True, axis = 0)


In [8]:
data["alcohol_freq"].value_counts()

alcohol_freq
Occasional    75161
Weekly        19833
Daily          5006
Name: count, dtype: int64

In [9]:
from sklearn.model_selection import train_test_split

In [10]:
data["annual_premium"].describe()

count    100000.000000
mean        582.320040
std         399.583722
min         211.670000
25%         352.070000
50%         463.585000
75%         666.697500
max       10962.550000
Name: annual_premium, dtype: float64

In [11]:
data["monthly_premium"].describe()

count    100000.000000
mean         48.526668
std          33.298640
min          17.640000
25%          29.340000
50%          38.630000
75%          55.560000
max         913.550000
Name: monthly_premium, dtype: float64

In [12]:
data2 = data.copy()
corr = data2.select_dtypes(include = "number").corr()["monthly_premium"].sort_values(ascending=False)

In [13]:
y = data["monthly_premium"]
data.drop("monthly_premium", inplace = True, axis = 1)
for idx, c in enumerate(corr):
    if c < 0.1:
        data2.drop(corr.index[idx], inplace = True, axis = 1)

In [14]:
data2.drop("monthly_premium", axis = 1, inplace = True)

In [15]:
x_train, x_test, y_train, y_test = train_test_split(data2, y, test_size = 0.2, random_state = 42)

In [16]:
x_train1 = x_train.copy()
x_train1["monthly_premium"] = y_train

In [17]:
corr = x_train1.select_dtypes(include = "number").corr()["monthly_premium"].sort_values(ascending=False)

In [18]:
corr

monthly_premium                1.000000
annual_premium                 1.000000
annual_medical_cost            0.965355
total_claims_paid              0.721703
avg_claim_amount               0.615001
risk_score                     0.295900
chronic_count                  0.287666
is_high_risk                   0.243532
days_hospitalized_last_3yrs    0.218634
hospitalizations_last_3yrs     0.198362
visits_last_year               0.193349
claims_count                   0.177359
hypertension                   0.149066
had_major_procedure            0.140803
systolic_bp                    0.139618
age                            0.128361
mental_health                  0.119420
arthritis                      0.117290
diastolic_bp                   0.115429
medication_count               0.109543
diabetes                       0.102677
Name: monthly_premium, dtype: float64

In [19]:
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

In [20]:
from sklearn.pipeline import make_pipeline

In [21]:
num_pip = make_pipeline(StandardScaler())
obj_pip = make_pipeline(OneHotEncoder(handle_unknown="ignore"))

In [22]:
preprocessing = make_column_transformer(
    (num_pip, make_column_selector(dtype_include = np.number)),
    (obj_pip, make_column_selector(dtype_include = object))
)

In [23]:
lin_reg = make_pipeline(preprocessing, LinearRegression())

In [24]:
lin_cv = -cross_val_score(lin_reg, x_train, y_train, cv = 3, scoring = "neg_root_mean_squared_error")

In [25]:
lin_cv

array([0.00291563, 0.00291088, 0.00290939])

In [26]:
y_train.describe()

count    80000.000000
mean        48.489099
std         33.312200
min         17.640000
25%         29.330000
50%         38.610000
75%         55.502500
max        913.550000
Name: monthly_premium, dtype: float64

In [27]:
lin_reg.fit(x_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('pipeline-1',
                                                  Pipeline(steps=[('standardscaler',
                                                                   StandardScaler())]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7d4039c69250>),
                                                 ('pipeline-2',
                                                  Pipeline(steps=[('onehotencoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7d4039fc5110>)])),
                ('linearregression', LinearRegression())])

In [28]:
lin_preds = lin_reg.predict(x_test)

In [29]:
from sklearn.metrics import mean_squared_error, r2_score

In [30]:
rmse = np.sqrt(mean_squared_error(y_test, lin_preds))

In [31]:
rmse

0.0029115099967396275

In [32]:
r2_score(y_test, lin_preds)

0.9999999923297043